# Generation Expansion Planning

Generation expansion planning lets the model decide whether to build candidate generators. A unit is a candidate when its investment cost is positive, that is `FixedInvestmentCost * FixedChargeRate > 0` in `oT_Data_Generation`.

The base 9-node case has no candidate generators, so we make the solar plant `SolarPV_1` a candidate and let the model decide whether to build it.

## 1. Set up a working copy of the case

In [1]:
import os, shutil
import pandas as pd

DIR = "work_GEP"          # parent folder that will hold the case
CaseName = "9n"      # we reuse the 9-node case from notebook 01

if os.path.exists(DIR):
    shutil.rmtree(DIR)
shutil.copytree(CaseName, os.path.join(DIR, CaseName))

# A coarse time resolution keeps the run fast for this tutorial.
param = os.path.join(DIR, CaseName, "oT_Data_Parameter_9n.csv")
df = pd.read_csv(param)
df.loc[:, "TimeStep"] = 24
df.to_csv(param, index=False)
print("Working copy of the 9n case is ready in", DIR)

Working copy of the 9n case is ready in work_GEP


## 2. Make SolarPV_1 a candidate and activate generation investment

Set `IndBinGenInvest = 1` in `oT_Data_Option` and ignore network investment. Then give `SolarPV_1` an investment cost (`FixedInvestmentCost` and an annual `FixedChargeRate`), allow it to be built (`InvestmentUp = 1`), and mark the decision as binary.

In [2]:
opt = pd.read_csv(os.path.join(DIR, CaseName, "oT_Data_Option_9n.csv"))
opt.loc[0, "IndBinGenInvest"] = 1
opt.loc[0, ["IndBinNetInvest", "IndBinGenRetirement"]] = 2
opt.to_csv(os.path.join(DIR, CaseName, "oT_Data_Option_9n.csv"), index=False)

gen = pd.read_csv(os.path.join(DIR, CaseName, "oT_Data_Generation_9n.csv"))
cand = gen[gen.columns[0]] == "SolarPV_1"
gen.loc[cand, "FixedInvestmentCost"] = 30   # MEUR
gen.loc[cand, "FixedChargeRate"]     = 0.05 # annualises the investment cost
gen.loc[cand, "InvestmentUp"]        = 1
gen.loc[cand, "InvestmentLo"]        = 0
gen.loc[cand, "BinaryInvestment"]    = "Yes"
gen.to_csv(os.path.join(DIR, CaseName, "oT_Data_Generation_9n.csv"), index=False)
gen.loc[cand, [gen.columns[0], "Technology", "FixedInvestmentCost", "FixedChargeRate", "InvestmentUp"]]

/var/folders/sw/46j89ccx613gt8sh72tlvl1w0000gn/T/ipykernel_45326/3763983977.py:12: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Yes' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  gen.loc[cand, "BinaryInvestment"]    = "Yes"


,Generator,Technology,FixedInvestmentCost,FixedChargeRate,InvestmentUp
14,SolarPV_1,RES,30.0,0.05,1.0


## 3. Run the model

In [3]:
from openTEPES.openTEPES import openTEPES_run

model = openTEPES_run(DIR, CaseName, "appsi_highs", "Yes", "No")
print("Total system cost [MEUR]:", round(model.vTotalSCost(), 3))

Input data                             ****
Reading the CSV files                  ...  0 s
Reading    input data                  ...  0 s


Setting up input data                  ...  1 s
Setting up variables                   ...  0 s
Total cost o.f.      model formulation ****
Investment elec      model formulation ****
Period 2030, Scenario sc01, Stage st1
Generation oper o.f. model formulation ****


Investment & operation var constraints ****
Inertia, oper resr, demand constraints ****
Storage   scheduling       constraints ****
Unit commitment            constraints ****


Ramp and min up/down time  constraints ****
Network    switching model constraints ****
Network    operation model constraints ****
Problem solving                        #### 1


Termination condition:  optimal
Problem solving with fixed investments #### 1


...

Writing elect network summary results  ...  0 s
Writing           reliability indexes  ...  0 s
Writing           flexibility results  ...  0 s


/private/tmp/claude-501/-Users-philias-ai-research-repos-openTEPES-tutorial/8f6d4ac5-9ff1-45bc-8a3c-5cc562abe3f1/scratchpad/venv312/lib/python3.12/site-packages/altair/utils/core.py:264: UserWarning: I don't know how to infer vegalite type from 'empty'.  Defaulting to nominal.
  warnings.warn(


Writing  generation operation results  ...  0 s
Writing         ESS operation results  ...  0 s
Writing elect netwk operation results  ...  0 s


Writing  marginal information results  ...  0 s


Writing              economic results  ...  0 s
Plotting electricity network     maps  ...  0 s
Total system cost [MEUR]: 197.32


## 4. Was the plant built?

The generation investment decision is written to `oT_Result_GenerationInvestment`. A value of 1 means `SolarPV_1` is built.

In [4]:
inv = pd.read_csv(os.path.join(DIR, CaseName, "oT_Result_GenerationInvestment_9n.csv"))
inv

,Period,SolarPV_1
0,2030,100.0
